In [1]:
import pandas as pd
import os

def get_train_years(year, n, data_dir='../../data/'):
    """
    Reads men's regular-season and NCAA tournament compact results from the local data directory,
    and returns two DataFrames for training:
      - reg_train: all regular-season games from (year-n) through year
      - tour_train: all tournament games from (year-n) through (year-1)
      
    Args:
        year (int): The year of the tournament to be predicted (e.g., 2025).
        n (int): The number of past seasons to include for training.
        data_dir (str): Relative or absolute path to the directory containing the CSV files.
        
    Returns:
        (pd.DataFrame, pd.DataFrame):
            reg_train, tour_train
            - reg_train: Regular-season games for years in [year-n, year].
            - tour_train: NCAA tournament games for years in [year-n, year-1].
    """
    # Read the data files
    reg_df = pd.read_csv(os.path.join(data_dir, 'MRegularSeasonCompactResults.csv'))
    tour_df = pd.read_csv(os.path.join(data_dir, 'MNCAATourneyCompactResults.csv'))
    
    # Identify which years to include
    train_years = list(range(year - n, year))  # for these years, use both regular season & tournament
    # For the 'current' year, we only use the regular season
    
    # Filter regular season data to [year-n ... year]
    reg_train = reg_df[reg_df['Season'].isin(train_years + [year])].copy()
    
    # Filter tournament data to [year-n ... year-1]
    tour_train = tour_df[tour_df['Season'].isin(train_years)].copy()
    
    return reg_train, tour_train


# ----------------
# Example usage:
if __name__ == "__main__":
    # Suppose we want to predict the 2025 tournament using the past 5 seasons
    year_to_predict = 2025
    n_past = 2
    
    reg_data, tour_data = get_train_years(year_to_predict, n_past)
    
    print("Regular Season Training Data:")
    print(reg_data.head())
    print(reg_data.shape)
    
    print("\nTournament Training Data:")
    print(tour_data.head())
    print(tour_data.shape)



Regular Season Training Data:
        Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
176080    2023       7     1101      65     1238      56    H      0
176081    2023       7     1103      81     1355      80    H      1
176082    2023       7     1104      75     1255      54    H      0
176083    2023       7     1112     117     1311      75    H      0
176084    2023       7     1113      62     1470      59    H      0
(15716, 8)

Tournament Training Data:
      Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
2384    2023     134     1338      60     1280      59    N      0
2385    2023     134     1394      75     1369      71    N      0
2386    2023     135     1113      98     1305      73    N      0
2387    2023     135     1192      84     1411      61    N      0
2388    2023     136     1104      96     1394      75    N      0
(134, 8)


In [2]:
import os
import pandas as pd

def merge_coaches_with_games(data_dir='../../data/'):
    """
    Reads MRegularSeasonCompactResults and MTeamCoaches, 
    merges the appropriate coach onto each game row (winning and losing side).
    
    Returns:
        DataFrame with the usual game columns plus:
            - 'WCoachName'
            - 'LCoachName'
    """
    # Read raw data
    reg_df = pd.read_csv(os.path.join(data_dir, 'MRegularSeasonCompactResults.csv'))
    coaches_df = pd.read_csv(os.path.join(data_dir, 'MTeamCoaches.csv'))
    
    # 1) Merge to find the winning team's coach.
    #    We'll merge on (Season, WTeamID -> TeamID) and then filter for DayNum in [FirstDayNum, LastDayNum].
    df_w = reg_df.merge(
        coaches_df, 
        left_on=['Season', 'WTeamID'], 
        right_on=['Season', 'TeamID'],
        how='left'
    )
    # Keep only rows where DayNum is within the coach's active range
    df_w = df_w[
        (df_w['DayNum'] >= df_w['FirstDayNum']) & 
        (df_w['DayNum'] <= df_w['LastDayNum'])
    ].copy()
    
    # Drop unneeded columns and rename the coach column
    df_w.drop(['TeamID','FirstDayNum','LastDayNum'], axis=1, inplace=True)
    df_w.rename(columns={'CoachName': 'WCoachName'}, inplace=True)
    
    # 2) Merge to find the losing team's coach similarly.
    df_l = reg_df.merge(
        coaches_df,
        left_on=['Season', 'LTeamID'],
        right_on=['Season', 'TeamID'],
        how='left'
    )
    df_l = df_l[
        (df_l['DayNum'] >= df_l['FirstDayNum']) & 
        (df_l['DayNum'] <= df_l['LastDayNum'])
    ].copy()
    df_l.drop(['TeamID','FirstDayNum','LastDayNum'], axis=1, inplace=True)
    df_l.rename(columns={'CoachName': 'LCoachName'}, inplace=True)
    
    # 3) Merge these two dataframes back together on the identifying columns
    game_cols = [
        'Season','DayNum','WTeamID','LTeamID',
        'WScore','LScore','WLoc','NumOT'
    ]
    # Keep the winning side's coach from df_w, and losing side's coach from df_l
    df_coaches_merged = df_w.merge(
        df_l[game_cols + ['LCoachName']], 
        on=game_cols, 
        how='inner'
    )
    
    # df_coaches_merged now has columns:
    #   Season, DayNum, WTeamID, LTeamID, WScore, LScore, WLoc, NumOT, WCoachName, LCoachName
    return df_coaches_merged


In [3]:
def compute_coach_elo(df, K=20, default_elo=1500):
    """
    Given a DataFrame of games that has columns:
        - Season, DayNum
        - WCoachName, LCoachName
      sorts them chronologically, and computes the Elo for each coach.
      
    Returns:
        A copy of df with 2 extra columns:
          - WCoachEloBefore
          - LCoachEloBefore
        which are the Elo ratings *before* the game was played.
        
      The function updates an internal dictionary that tracks each coach's Elo rating
      across all seasons in chronological order.
    """
    # We will return a new DataFrame so as not to mutate the original.
    df = df.copy()
    
    # Sort by Season, then DayNum (if you have multiple seasons in the dataset).
    df.sort_values(by=['Season','DayNum'], inplace=True)
    
    # Dictionary for storing current Elo of each coach
    coach_elo = {}
    
    # New columns to store Elo *before* the game
    df['WCoachEloBefore'] = 0.0
    df['LCoachEloBefore'] = 0.0
    
    for idx, row in df.iterrows():
        w_coach = row['WCoachName']
        l_coach = row['LCoachName']
        
        # If a coach has never been seen, assign default Elo
        if w_coach not in coach_elo:
            coach_elo[w_coach] = default_elo
        if l_coach not in coach_elo:
            coach_elo[l_coach] = default_elo
        
        # Current Elo for each coach (before this game)
        w_elo_before = coach_elo[w_coach]
        l_elo_before = coach_elo[l_coach]
        
        # Store them in the DataFrame
        df.at[idx, 'WCoachEloBefore'] = w_elo_before
        df.at[idx, 'LCoachEloBefore'] = l_elo_before
        
        # Elo expected scores
        expected_w = 1.0 / (1.0 + 10 ** ((l_elo_before - w_elo_before)/400))
        expected_l = 1.0 - expected_w
        
        # The winning coach gets a score of 1, losing coach gets 0
        score_w = 1.0
        score_l = 0.0
        
        # Update the Elo
        coach_elo[w_coach] = w_elo_before + K * (score_w - expected_w)
        coach_elo[l_coach] = l_elo_before + K * (score_l - expected_l)
        
    return df


In [4]:
def compute_team_losses(df):
    """
    Given a DataFrame of games with columns:
      - Season, DayNum
      - WTeamID, LTeamID
    returns a copy of df with 2 extra columns:
      - WTeamPrevLosses
      - LTeamPrevLosses
    which represent the number of losses each team had *before* the current game.
    
    Assumes df is sorted by (Season, DayNum).
    """
    df = df.copy()
    
    # For storing number of losses so far: key = (Season, TeamID)
    losses_count = {}
    
    df['WTeamPrevLosses'] = 0
    df['LTeamPrevLosses'] = 0
    
    for idx, row in df.iterrows():
        season = row['Season']
        w_team = row['WTeamID']
        l_team = row['LTeamID']
        
        # If not in dictionary, initialize
        if (season, w_team) not in losses_count:
            losses_count[(season, w_team)] = 0
        if (season, l_team) not in losses_count:
            losses_count[(season, l_team)] = 0
        
        # Store the number of losses so far
        df.at[idx, 'WTeamPrevLosses'] = losses_count[(season, w_team)]
        df.at[idx, 'LTeamPrevLosses'] = losses_count[(season, l_team)]
        
        # The losing team increments its loss count
        losses_count[(season, l_team)] += 1
    
    return df


In [5]:
def build_features_for_training(data_dir='../../data/'):
    """
    Example pipeline:
      1) Merge MRegularSeasonCompactResults with MTeamCoaches to identify WCoachName, LCoachName.
      2) Compute coach Elo rating across all games (chronological).
      3) Track the number of previous losses for each team.
      
    Returns:
        DataFrame with columns:
          - Season, DayNum, WTeamID, LTeamID, WScore, LScore, WLoc, NumOT
          - WCoachName, LCoachName
          - WCoachEloBefore, LCoachEloBefore
          - WTeamPrevLosses, LTeamPrevLosses
    """
    # Step 1: Merge coaches with game data
    df = merge_coaches_with_games(data_dir)
    
    # Step 2: Compute coach Elo
    df = compute_coach_elo(df, K=20, default_elo=1500)
    
    # Step 3: Track team losses
    # (Note: we need it sorted by Season, DayNum again, because compute_coach_elo also sorts.)
    df = compute_team_losses(df)
    
    return df

# ---------------------
# Example usage:
if __name__ == "__main__":
    final_df = build_features_for_training(data_dir='../../data/')
    print(final_df.head(10))
    
    # final_df now contains the extra columns:
    #   WCoachName, LCoachName, WCoachEloBefore, LCoachEloBefore,
    #   WTeamPrevLosses, LTeamPrevLosses
    #
    # You can then use these features (plus others you engineer) in your model
    # to predict tournament outcomes.


   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  \
0    1985      20     1228      81     1328      64    N      0   
1    1985      25     1106      77     1354      70    H      0   
2    1985      25     1112      63     1223      56    H      0   
3    1985      25     1165      70     1432      54    H      0   
4    1985      25     1192      86     1447      74    H      0   
5    1985      25     1218      79     1337      78    H      0   
6    1985      25     1228      64     1226      44    N      0   
7    1985      25     1242      58     1268      56    N      0   
8    1985      25     1260      98     1133      80    H      0   
9    1985      25     1305      97     1424      89    H      0   

      WCoachName       LCoachName  WCoachEloBefore  LCoachEloBefore  \
0     lou_henson      billy_tubbs           1500.0           1500.0   
1   james_oliver   chico_caldwell           1500.0           1500.0   
2     lute_olson         gene_iba           1500.

In [6]:
def build_features_for_training(data_dir='../../data/'):
    """
    Example pipeline that:
      1) Merges MRegularSeasonCompactResults with MTeamCoaches
      2) Computes Coach Elo
      3) Tracks each team's prior losses.
    Returns a DataFrame with the added feature columns.
    """
    # 1) Merge coaches
    df = merge_coaches_with_games(data_dir)  # from earlier example
    # 2) Compute coach Elo
    df = compute_coach_elo(df, K=20, default_elo=1500)
    # 3) Track team losses
    df = compute_team_losses(df)
    return df


In [7]:
final_df = build_features_for_training(data_dir='../../data/')


In [8]:
import pandas as pd

def create_train_rows(final_df):
    """
    Given final_df (one row per game, from winner perspective),
    create two rows per game:
      - Row A: Team1 = winner, Team2 = loser, target=1
      - Row B: Team1 = loser,  Team2 = winner, target=0

    Also create features like:
      'elo_diff' = Team1CoachEloBefore - Team2CoachEloBefore
      'loss_diff' = Team1PrevLosses - Team2PrevLosses

    Returns a new DataFrame with columns:
      [Season, DayNum, Team1ID, Team2ID, elo_diff, loss_diff, target]
    """
    rows_list = []
    
    for idx, row in final_df.iterrows():
        # Extract from original row
        season = row['Season']
        daynum = row['DayNum']
        
        w_id = row['WTeamID']
        l_id = row['LTeamID']
        
        w_elo = row['WCoachEloBefore']
        l_elo = row['LCoachEloBefore']
        
        w_loss = row['WTeamPrevLosses']
        l_loss = row['LTeamPrevLosses']
        
        # Row A: Team1 = winner
        rowA = {
            'Season': season,
            'DayNum': daynum,
            'Team1ID': w_id,
            'Team2ID': l_id,
            'elo_diff': w_elo - l_elo,
            'loss_diff': w_loss - l_loss,
            'target': 1
        }
        
        # Row B: Team1 = loser
        rowB = {
            'Season': season,
            'DayNum': daynum,
            'Team1ID': l_id,
            'Team2ID': w_id,
            'elo_diff': l_elo - w_elo,
            'loss_diff': l_loss - w_loss,
            'target': 0
        }
        
        rows_list.append(rowA)
        rows_list.append(rowB)
        
    new_df = pd.DataFrame(rows_list)
    return new_df


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

def train_and_compare_models(data_dir='../../data/'):
    # 1) Build features from historical regular-season data
    final_df = build_features_for_training(data_dir)
    
    # 2) Convert to two-rows-per-game
    train_df = create_train_rows(final_df)
    
    # 3) Split into features (X) and target (y)
    X = train_df[['elo_diff', 'loss_diff']]
    y = train_df['target']
    
    # 4) Train/test split (random for example)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # -------------------------------
    # Model A: Logistic Regression
    # -------------------------------
    logreg = LogisticRegression()
    logreg.fit(X_train, y_train)
    
    # Probability that Team1 wins
    y_prob_logreg = logreg.predict_proba(X_test)[:,1]
    brier_logreg = brier_score_loss(y_test, y_prob_logreg)
    
    # -------------------------------
    # Model B: Random Forest
    # -------------------------------
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    y_prob_rf = rf.predict_proba(X_test)[:,1]
    brier_rf = brier_score_loss(y_test, y_prob_rf)
    
    print("Logistic Regression Brier Score:", brier_logreg)
    print("Random Forest Brier Score:      ", brier_rf)

    return logreg, rf


In [10]:
def evaluate_on_tournament(model, season, data_dir='../../data/'):
    """
    Evaluate a trained model on the NCAA Tournament games for a specific season.
    We'll compute the Brier score for the model's predictions.
    """
    # 1) Load NCAA Tourney data for that season
    tour_df = pd.read_csv(os.path.join(data_dir, 'MNCAATourneyCompactResults.csv'))
    # Filter to the desired season
    tour_df = tour_df[tour_df['Season'] == season].copy()
    
    # 2) Merge with coaches, compute Elo, etc. just like for regular season
    #    (In practice, you want the Elo "carried over" from the end of the regular season,
    #     so you might do something more sophisticated than re-calculating from scratch.)
    #    For simplicity, let's assume you have a function that returns a tournament DF
    #    with the same columns as final_df.
    #    Or you might adapt build_features_for_training to handle tournament data as well.
    
    # Example (pseudocode):
    #   tour_features_df = build_features_for_tourney(season, data_dir)
    #   # which merges coaches, daynum, etc.
    
    # For brevity, let's pretend we have that in 'tour_final_df':
    # tour_final_df = ...
    
    # We'll skip the details here. Suppose we have:
    #   WCoachEloBefore, LCoachEloBefore, WTeamPrevLosses, LTeamPrevLosses, etc.
    
    # 3) Convert to two-row format
    # tour_train_df = create_train_rows(tour_final_df)
    
    # 4) Predict
    # X_tourney = tour_train_df[['elo_diff','loss_diff']]
    # y_tourney = tour_train_df['target']
    # y_prob = model.predict_proba(X_tourney)[:,1]
    
    # 5) Brier Score
    # brier = brier_score_loss(y_tourney, y_prob)
    # print(f"Brier score for season {season}: {brier}")
    
    # For demonstration, let's just print a placeholder:
    print(f"(Pseudo) Evaluating model on {season} tournament games... [details omitted]")


In [ ]:
from sklearn.metrics import brier_score_loss

# Suppose y_true are the actual outcomes (0 or 1),
# and y_prob are your predicted probabilities for "1".
brier = brier_score_loss(y_true, y_prob)
print("Brier Score:", brier)


NameError: name 'y_true' is not defined

: 